In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

E:\LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [9]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode() #Encode the image into base64 and pass it the qwen/qwen3.6 model (vision model)..because the model need some datatype to extract text the iamge, before llm we use OCR

llm = ChatGroq(model="qwen/qwen3.6-27b")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

#print(message)
response = llm.invoke([message])
print(response.content)


<think>
The user wants me to extract data from a blood test report image.

**1. Analyze the input image:**
- It's a lab report for a patient named Rajesh Sharma.
- Date: May 7, 2026.
- There are sections: COMPLETE BLOOD COUNT (CBC), LIPID PANEL, METABOLIC PANEL, and LIVER FUNCTION (header only).
- I need to read the values and the reference ranges (Normal ranges).
- I need to compare values to ranges and flag abnormalities.

**2. Extract Data Section by Section:**

*   **Patient Info:**
    *   Name: Rajesh Sharma
    *   Age: 48
    *   Sex: Male
    *   Date: May 7, 2026

*   **COMPLETE BLOOD COUNT (CBC):**
    *   Hemoglobin: 15.1 g/dL. Normal range: 13.5 - 17.5. -> **Normal**
    *   Hematocrit: 44%. Normal range: 41 - 53%. -> **Normal**
    *   WBC: 6.8 x 10^3/uL. Normal range: 4.5 - 11.0. -> **Normal**
    *   Platelets: 220 x 10^3/uL. Normal range: 150 - 400. -> **Normal**

*   **LIPID PANEL:**
    *   Total Cholesterol: 238 mg/dL. Normal range: <200. -> **High (Abnormal)**
   

In [5]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [6]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [7]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

Based on the blood work report provided:

**Lipid Panel Analysis:**
*   **Total Cholesterol:** 238 mg/dL (Normal: <200) - **High**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) - **High**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) - **Low**
*   **Triglycerides:** 188 mg/dL (Normal: <150) - **High**

**Metabolic Panel Analysis:**
*   **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) - Normal
*   **HbA1c:** 5.3% (Normal: <5.7%) - Normal

**Conclusion:**
The patient's blood sugar levels are within the normal range. However, the lipid profile shows high total cholesterol, high LDL ("bad" cholesterol), low HDL ("good" cholesterol), and high triglycerides. This indicates a condition of **high cholesterol**.

Here is a suggested diet plan for managing high cholesterol:

**Diet Plan for High Cholesterol**

**Foods to Eat:**
*   **Fruits and Vegetables:** Rich in fiber and antioxidants.
*   **Whole Grains:** Oats, brown rice, and whole wheat help lower cholesterol.
*   **Lean Protein:** 